# 🛠️ Chapter 02-02b: Hands-On — CoT, Structured Outputs & Multi-Step Reasoning

---

## Time to Deploy the Advanced Arsenal! ⚔️

In the concept notebook, you learned about CoT, ToT, Self-Consistency, and structured outputs. Now let's **code every one of them**.

By the end of this notebook, you'll have:
- ✅ Implemented Chain-of-Thought and compared it to zero-shot
- ✅ Built a Tree-of-Thought problem solver
- ✅ Used Self-Consistency (majority voting)
- ✅ Mastered Pydantic structured outputs
- ✅ Built a multi-step reasoning system

Let's go! 🚀

---

## 📦 Setup

In [ ]:
# !pip install openai langchain-openai langchain-core python-dotenv pydantic

In [ ]:
import os
import json
from collections import Counter
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field

load_dotenv()

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
llm_creative = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)  # For self-consistency

api_key = os.getenv("OPENAI_API_KEY")
print(f"✅ Ready!" if api_key else "❌ Set OPENAI_API_KEY in .env")

---

## 🧪 Exercise 1: Chain-of-Thought vs Zero-Shot

Let's prove that CoT actually works with a hard math problem.

In [ ]:
# Hard math word problem
problem = """A store has 45 apples. They sell 12 on Monday, buy 28 more on Tuesday, 
sell half of what they have on Wednesday, and then receive a shipment of 15 on Thursday. 
How many apples do they have at the end of Thursday?"""

# ZERO-SHOT: Just ask for the answer
zero_shot = f"""Solve this math problem. Give only the final number.

{problem}

Answer:"""

result_zero = llm.invoke(zero_shot)
print("❌ ZERO-SHOT (no reasoning):")
print(f"   {result_zero.content}")

print("\n" + "=" * 60)

# CHAIN-OF-THOUGHT: Ask for step-by-step reasoning
cot_prompt = f"""Solve this math problem step by step. Show ALL your work.

{problem}

Let's think step by step:"""

result_cot = llm.invoke(cot_prompt)
print("\n✅ CHAIN-OF-THOUGHT (with reasoning):")
print(f"   {result_cot.content}")

# Correct answer: 45 - 12 = 33, 33 + 28 = 61, 61/2 = 30.5 → 30 or 31, 30.5 + 15 = 45.5
print("\n📐 Manual calculation: 45 → 33 → 61 → 30.5 → 45.5")

### 💡 Key Insight:

> The CoT version shows its work, which means: (1) you can verify its reasoning, (2) it catches its own mistakes, and (3) the final answer is more likely correct.

---

## 🧪 Exercise 2: Few-Shot CoT (With Worked Examples)

In [ ]:
# Few-shot CoT: Give examples WITH reasoning steps

few_shot_cot = ChatPromptTemplate.from_messages([
    ("system", """You are a logic expert. Solve problems step by step.
Follow the exact reasoning format shown in the examples."""),
    ("human", """Example 1:
Q: A cafeteria had 23 apples. They used 20 to make lunch and bought 6 more. How many apples do they have?
A: Step 1: Started with 23 apples.
   Step 2: Used 20 for lunch → 23 - 20 = 3.
   Step 3: Bought 6 more → 3 + 6 = 9.
   Final answer: 9

Example 2:
Q: Roger has 5 tennis balls. He buys 2 cans of 3 balls each. How many does he have?
A: Step 1: Roger starts with 5 balls.
   Step 2: He buys 2 cans × 3 balls = 6 balls.
   Step 3: Total = 5 + 6 = 11.
   Final answer: 11

Now solve:
Q: {question}
A:""")
])

chain = few_shot_cot | llm

result = chain.invoke({
    "question": "A library had 156 books. They donated 45 to a school, received 72 new books, "
                "then lost 8 in a flood. How many books do they have?"
})

print("📚 FEW-SHOT CoT Result:")
print(result.content)
print("\n📐 Expected: 156 - 45 + 72 - 8 = 175")

---

## 🧪 Exercise 3: Tree-of-Thought

The model considers MULTIPLE approaches before picking the best one.

In [ ]:
# Tree-of-Thought: Explore multiple solutions

tot_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an expert problem solver who considers multiple approaches.
For each problem:
1. Propose EXACTLY 3 different approaches
2. Evaluate each approach (pros, cons, confidence 1-10)
3. Pick the best approach and execute it
4. Give the final answer"""),
    ("human", "{problem}")
])

tot_chain = tot_prompt | llm

result = tot_chain.invoke({
    "problem": """A startup has $50,000 to spend on marketing. 
They want to maximize user acquisition for their B2B SaaS product targeting small businesses. 
What's the best marketing strategy?"""
})

print("🌳 TREE-OF-THOUGHT Result:")
print(result.content)

### 💡 Key Insight:

> ToT gives you **a structured comparison** instead of the model jumping to one solution. This is especially valuable when decisions have trade-offs and there's no single "correct" answer.

---

## 🧪 Exercise 4: Self-Consistency (Majority Voting)

In [ ]:
# Self-Consistency: Ask N times, take majority vote
# Using temperature > 0 to get diverse reasoning paths

tricky_problem = """If it takes 5 machines 5 minutes to make 5 widgets, 
how long would it take 100 machines to make 100 widgets?"""

prompt = f"""Solve this carefully. Think step by step, then give your final answer 
as JUST a number followed by 'minutes'.

{tricky_problem}

Final answer:"""

N_SAMPLES = 5
answers = []

print(f"🎲 SELF-CONSISTENCY: Asking {N_SAMPLES} times...\n")

for i in range(N_SAMPLES):
    result = llm_creative.invoke(prompt)  # temperature=0.7 for diversity
    answer = result.content.strip()
    answers.append(answer)
    print(f"  Sample {i+1}: {answer}")

# Count votes
vote_counts = Counter(answers)
winner = vote_counts.most_common(1)[0]

print(f"\n📊 VOTE RESULTS: {dict(vote_counts)}")
print(f"🏆 MAJORITY ANSWER: {winner[0]} (got {winner[1]}/{N_SAMPLES} votes)")
print(f"\n📐 Correct answer: 5 minutes (each machine makes 1 widget in 5 min)")

---

## 🧪 Exercise 5: Pydantic Structured Output

This is the **production-grade** way to get structured data from LLMs.

In [ ]:
from pydantic import BaseModel, Field
from typing import Optional

# Step 1: Define your schema as a Pydantic model
class MovieReview(BaseModel):
    """Structured analysis of a movie review."""
    movie_title: str = Field(description="Title of the movie")
    sentiment: str = Field(description="positive, negative, or mixed")
    rating: float = Field(description="Rating from 1.0 to 10.0")
    key_themes: list[str] = Field(description="Main themes or topics mentioned")
    recommended: bool = Field(description="Whether the reviewer recommends it")
    one_line_summary: str = Field(description="One sentence summary")

# Step 2: Bind the schema to the model
structured_llm = llm.with_structured_output(MovieReview)

# Step 3: Invoke — returns a Pydantic object, NOT a string!
review_text = """Just watched Interstellar again and it blew my mind even more the second time. 
The visuals are stunning, Hans Zimmer's score gives me chills, and the science (while stretched) 
is fascinating. The father-daughter relationship is the emotional core that makes you cry. 
Only complaint: some dialogue is hard to hear. 9/10 would absolutely recommend."""

result = structured_llm.invoke(f"Analyze this movie review: {review_text}")

print("🎬 STRUCTURED OUTPUT (Pydantic):")
print(f"   Type: {type(result).__name__}")
print(f"   Movie: {result.movie_title}")
print(f"   Sentiment: {result.sentiment}")
print(f"   Rating: {result.rating}/10")
print(f"   Themes: {result.key_themes}")
print(f"   Recommended: {'✅ Yes' if result.recommended else '❌ No'}")
print(f"   Summary: {result.one_line_summary}")

# Access as a dict
print(f"\n   📦 As dict: {result.model_dump()}")

### 💡 Why This Is Powerful:

```
┌─────────────────────────────────────────────────────────────────────┐
│  Before Pydantic: result = json.loads(response.content) 😬        │
│    → Might fail! Might have wrong keys! Might have wrong types!   │
│                                                                     │
│  After Pydantic:  result = structured_llm.invoke(prompt) 😎       │
│    → GUARANTEED correct schema                                     │
│    → Type-safe: result.rating is a float, not a string            │
│    → IDE autocomplete works!                                       │
│    → Validation built in                                           │
└─────────────────────────────────────────────────────────────────────┘
```

---

## 🧪 Exercise 6: Complex Structured Output with Nested Models

In [ ]:
# Nested Pydantic models for complex data extraction

class TechStack(BaseModel):
    """Individual technology in a job posting."""
    name: str = Field(description="Technology name")
    category: str = Field(description="frontend, backend, database, devops, or other")
    required: bool = Field(description="True if required, False if nice-to-have")

class JobPosting(BaseModel):
    """Structured extraction from a job posting."""
    title: str = Field(description="Job title")
    seniority: str = Field(description="junior, mid, senior, lead, or staff")
    salary_min: Optional[int] = Field(description="Minimum salary if mentioned")
    salary_max: Optional[int] = Field(description="Maximum salary if mentioned")
    remote: bool = Field(description="Whether remote work is available")
    tech_stack: list[TechStack] = Field(description="Technologies mentioned")
    years_experience: Optional[int] = Field(description="Years of experience required")
    summary: str = Field(description="1-2 sentence summary")

# Bind to LLM
job_extractor = llm.with_structured_output(JobPosting)

# Test with a messy job posting
job_text = """We're looking for a Senior Full-Stack Engineer to join our growing team!
Must have 5+ years experience with React, TypeScript, and Node.js. 
Experience with PostgreSQL and AWS is required. Knowledge of GraphQL and Docker is a plus.
Salary range: $150K-$200K. Fully remote position. 
You'll be building our next-gen AI-powered analytics platform."""

result = job_extractor.invoke(f"Extract information from this job posting:\n{job_text}")

print("💼 JOB POSTING EXTRACTION:")
print(f"   Title: {result.title}")
print(f"   Level: {result.seniority}")
print(f"   Salary: ${result.salary_min:,}-${result.salary_max:,}" if result.salary_min else "   Salary: Not specified")
print(f"   Remote: {'🏠 Yes' if result.remote else '🏢 No'}")
print(f"   Experience: {result.years_experience}+ years")
print(f"   Summary: {result.summary}")
print(f"\n   🔧 Tech Stack:")
for tech in result.tech_stack:
    req = "✅ Required" if tech.required else "⭐ Nice-to-have"
    print(f"      {tech.name} ({tech.category}) — {req}")

---

## 🚀 Build: Multi-Step Reasoning System

Let's combine CoT + structured output into a real system that:
1. Takes a complex question
2. Breaks it down into sub-questions
3. Answers each sub-question
4. Synthesizes a final answer

In [ ]:
# Step 1: Break complex question into sub-questions

class QuestionBreakdown(BaseModel):
    """Breaking a complex question into sub-questions."""
    original_question: str = Field(description="The original question")
    sub_questions: list[str] = Field(description="3-5 simpler sub-questions to answer first")
    reasoning: str = Field(description="Why these sub-questions help answer the main question")

decomposer = llm.with_structured_output(QuestionBreakdown)

complex_question = """Should a startup with a team of 5 engineers switch from a monolithic 
Python Django app to microservices using Go, considering they have 10K daily active users 
and are growing 20% month over month?"""

breakdown = decomposer.invoke(
    f"Break this complex question into simpler sub-questions:\n{complex_question}"
)

print("🔍 QUESTION DECOMPOSITION")
print(f"\nOriginal: {breakdown.original_question}")
print(f"\nSub-questions:")
for i, q in enumerate(breakdown.sub_questions, 1):
    print(f"  {i}. {q}")
print(f"\nReasoning: {breakdown.reasoning}")

In [ ]:
# Step 2: Answer each sub-question + synthesize

class SubAnswer(BaseModel):
    """Answer to a sub-question."""
    question: str
    answer: str
    confidence: float = Field(description="0.0 to 1.0")

class FinalAnalysis(BaseModel):
    """Synthesized final analysis."""
    recommendation: str = Field(description="yes, no, or it depends")
    confidence: float
    reasoning: str
    risks: list[str]
    next_steps: list[str]

# Answer sub-questions
sub_answers = []
answerer = llm.with_structured_output(SubAnswer)

print("📝 ANSWERING SUB-QUESTIONS:\n")
for q in breakdown.sub_questions:
    answer = answerer.invoke(f"Answer concisely: {q}")
    sub_answers.append(answer)
    print(f"  Q: {answer.question}")
    print(f"  A: {answer.answer}")
    print(f"  Confidence: {answer.confidence}")
    print()

# Synthesize into final answer
synthesizer = llm.with_structured_output(FinalAnalysis)

context = "\n".join([f"Q: {a.question}\nA: {a.answer}" for a in sub_answers])

final = synthesizer.invoke(
    f"""Based on these sub-answers, provide a final analysis for: {complex_question}
    
Sub-answers:
{context}"""
)

print("=" * 60)
print("\n🎯 FINAL ANALYSIS")
print(f"   Recommendation: {final.recommendation.upper()}")
print(f"   Confidence: {final.confidence}")
print(f"   Reasoning: {final.reasoning}")
print(f"\n   ⚠️ Risks:")
for r in final.risks:
    print(f"      • {r}")
print(f"\n   ✅ Next Steps:")
for s in final.next_steps:
    print(f"      • {s}")

---

## ✍️ Practice: Build a Debate Simulator

**Challenge:** Create a system where 2 LLM "experts" debate a topic and a "judge" picks the winner.

In [ ]:
# YOUR CODE HERE

# TODO: Define Pydantic models:
#   class Argument(BaseModel): position, key_points, evidence
#   class DebateResult(BaseModel): winner, reasoning, score_side_a, score_side_b

# TODO: Create 2 LLM calls with different system prompts:
#   Expert A: "You argue FOR..." 
#   Expert B: "You argue AGAINST..."

# TODO: Judge LLM evaluates both arguments

# topic = "AI will replace most programming jobs within 10 years"
# Test your debate simulator!

<details>
<summary>💡 Click to see solution</summary>

```python
class Argument(BaseModel):
    position: str
    key_points: list[str]
    strongest_evidence: str

class DebateResult(BaseModel):
    winner: str = Field(description="'side_a' or 'side_b'")
    reasoning: str
    score_a: int = Field(description="Score 1-10")
    score_b: int = Field(description="Score 1-10")

topic = "AI will replace most programming jobs within 10 years"

# Expert A: FOR
debater_a = llm.with_structured_output(Argument)
arg_a = debater_a.invoke(f"Argue FOR this position: {topic}")

# Expert B: AGAINST  
debater_b = llm.with_structured_output(Argument)
arg_b = debater_b.invoke(f"Argue AGAINST this position: {topic}")

# Judge
judge = llm.with_structured_output(DebateResult)
verdict = judge.invoke(
    f"Side A (FOR): {arg_a.model_dump()}\nSide B (AGAINST): {arg_b.model_dump()}\n"
    f"Judge this debate fairly."
)

print(f"Winner: {verdict.winner} ({verdict.score_a} vs {verdict.score_b})")
```

</details>

---

## 💡 Interview Prep: Common Questions

```
╔═══════════════════════════════════════════════════════════════════════╗
║           INTERVIEW QUESTIONS FOR THIS MODULE                        ║
╠═══════════════════════════════════════════════════════════════════════╣
║                                                                       ║
║  Q: How would you implement structured output in production?         ║
║  A: Use LangChain's with_structured_output() with Pydantic models.  ║
║     Define your schema as a Pydantic BaseModel, bind it to the LLM. ║
║     The model returns validated Python objects, not raw strings.     ║
║     Add retry logic for edge cases. Use Optional fields sensibly.   ║
║                                                                       ║
║  Q: What is self-consistency? When would you use it?                 ║
║  A: Running the same prompt N times with temperature > 0 and taking ║
║     a majority vote on the answers. Use for high-stakes decisions   ║
║     (medical, financial, safety). Trade-off: costs N× more tokens.  ║
║                                                                       ║
║  Q: How would you combine CoT with structured output?               ║
║  A: Define a Pydantic model with a 'reasoning_steps' field          ║
║     (list[str]) AND a 'final_answer' field. The model is forced     ║
║     to articulate its thinking AND give a structured result.        ║
║                                                                       ║
╚═══════════════════════════════════════════════════════════════════════╝
```

---

## 📝 Quick Quiz

---

### Question 1:
What does `llm.with_structured_output(MyModel)` return when invoked?

- A) A JSON string
- B) A Python dictionary
- C) A Pydantic model instance (MyModel object)
- D) An AIMessage object

<details><summary>Answer</summary>

**C) A Pydantic model instance (MyModel object)**

The `with_structured_output` method returns an instance of the Pydantic model you passed in. You can access fields with dot notation (`result.sentiment`), call `result.model_dump()` for a dict, or `result.model_dump_json()` for a JSON string.

</details>

---

### Question 2:
You're running self-consistency with N=5 and getting 5 different answers. What should you try?

- A) Increase N to 100
- B) Lower temperature and/or improve the prompt
- C) Switch to a smaller model
- D) Give up

<details><summary>Answer</summary>

**B) Lower temperature and/or improve the prompt**

If 5 runs give 5 different answers, the model is too uncertain. Either the temperature is too high (try 0.3-0.5 instead of 1.0) or the prompt is ambiguous and needs clarification. Increasing N won't help if the model doesn't converge.

</details>

---

### Question 3:
Which is the correct order for a multi-step reasoning system?

- A) Answer → Decompose → Synthesize
- B) Decompose → Answer sub-questions → Synthesize
- C) Synthesize → Decompose → Answer
- D) Just use one big prompt

<details><summary>Answer</summary>

**B) Decompose → Answer sub-questions → Synthesize**

The pattern is: break a complex question into simpler parts, answer each part independently, then combine the sub-answers into a final analysis. This is more reliable than trying to answer everything in one prompt.

</details>

---

## ✅ Key Takeaways

```
┌─────────────────────────────────────────────────────────────────────┐
│  📌 CoT: "Think step by step" proves reasoning = better answers.  │
├─────────────────────────────────────────────────────────────────────┤
│  📌 ToT: 3 approaches → evaluate → pick best. Great for planning. │
├─────────────────────────────────────────────────────────────────────┤
│  📌 Self-Consistency: Ask N times, majority vote. High stakes.     │
├─────────────────────────────────────────────────────────────────────┤
│  📌 with_structured_output + Pydantic = production gold standard.  │
├─────────────────────────────────────────────────────────────────────┤
│  📌 Multi-step: decompose → answer parts → synthesize.            │
├─────────────────────────────────────────────────────────────────────┤
│  📌 Nested Pydantic models handle complex data extraction.         │
└─────────────────────────────────────────────────────────────────────┘
```

---

## ➡️ Next Up: LangChain Core!

Chapter 3: Chat Models, LCEL, Chains, and Output Parsers. We're going from prompting to **building real applications** with LangChain!

---

### 🎉 You've Mastered Advanced Prompting!
CoT, ToT, Self-Consistency, Pydantic structured output, multi-step reasoning — you're ready for production! 🚀